# 24. Live Tutorial 3: Model Reduction and Fractional Designs

A screening design (Notebook 18) deliberately writes down more candidate
terms than are likely to survive — main effects, plus every interaction a
process engineer has a physical reason to suspect. This session is about
what happens *after* those runs come back: turning an over-specified
candidate model into a defensible, reduced one, and understanding exactly
what you're trusting (and risking) when you do it on a design that isn't
full resolution.

This session builds directly on Notebook 12 (Live Tutorial 2, Statistics)'s
backward elimination and Notebook 18's aliasing theory rather than
introducing either from scratch, so it moves faster than this course's
other live tutorials.

**Session plan**
1. Recap: why screening needs model reduction
2. Model reduction: from a candidate list to a defensible model
3. Fractional factorial, revisited through a reduction lens

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, combinations
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import pyDOE3

sns.set_theme(style='ticks', palette='colorblind')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
rng = np.random.default_rng(303)

## 24.1 Recap: Why Screening Needs Model Reduction

A screening design (Notebook 18; Notebook 23 §23.7) exists to answer *"which
of many candidate factors actually matter?"* cheaply — which means you
deliberately write down **more candidate terms than you expect to survive**
(main effects, plausible interactions the process engineer suspects). The
raw fitted model is therefore usually cluttered with terms that are not
really significant. **Model reduction** — Notebook 12 (Live Tutorial 2, Statistics)
§12.5 introduced backward elimination for ordinary regression — is exactly
as necessary here, and the mechanics are identical, because a factorial
ANOVA *is* a regression (that notebook's central point) on coded
columns.

## 24.2 Model Reduction: From a Candidate List to a Defensible Model

**Adhesive bonding study**: five process factors, lap-shear strength
(MPa) response.

| Factor | Low ($-1$) | High ($+1$) |
|---|---|---|
| A — Surface treatment time | 30 s | 90 s |
| B — Cure temperature | 60°C | 100°C |
| C — Clamp pressure | 0.5 MPa | 2.0 MPa |
| D — Bondline thickness | 0.1 mm | 0.3 mm |
| E — Humidity during cure | 30% RH | 70% RH |

A $2^{5-1}$ half-fraction (generator $I=ABCDE$, Resolution V — Notebook 18's
construction) gives 16 runs. Rather than fit *every* possible interaction
(most would be unestimable — only 16 runs total), the process engineer
writes down a **candidate list**: all 5 main effects, plus 4 interactions
she has a physical reason to suspect.

In [2]:
design_gen = pyDOE3.fracfact('a b c d abcd')     # 2^(5-1), Resolution V, I=ABCDE
df_adh = pd.DataFrame(design_gen, columns=['A', 'B', 'C', 'D', 'E'])

# ── True model: some candidate terms are real, two are included but null ──────
true_strength = (18
                  + 4*df_adh.A + 6*df_adh.B + 3*df_adh.C - 2*df_adh.D - 1*df_adh.E
                  + 2*df_adh.A*df_adh.B          # genuine: longer treatment helps MORE at high cure temp
                  - 1.5*df_adh.C*df_adh.D        # genuine: pressure matters less as bondline thins
                  + 0*df_adh.B*df_adh.C          # candidate, but NOT real
                  + 0*df_adh.A*df_adh.E)         # candidate, but NOT real
df_adh['strength_MPa'] = (true_strength + rng.normal(0, 1.0, len(df_adh))).round(2)
print(df_adh)

      A    B    C    D    E  strength_MPa
0  -1.0 -1.0 -1.0 -1.0  1.0          5.73
1  -1.0 -1.0 -1.0  1.0 -1.0          7.05
2  -1.0 -1.0  1.0 -1.0 -1.0         16.63
3  -1.0 -1.0  1.0  1.0  1.0          8.74
4  -1.0  1.0 -1.0 -1.0 -1.0         15.82
5  -1.0  1.0 -1.0  1.0  1.0         15.12
6  -1.0  1.0  1.0 -1.0  1.0         22.45
7  -1.0  1.0  1.0  1.0 -1.0         17.49
8   1.0 -1.0 -1.0 -1.0 -1.0         11.18
9   1.0 -1.0 -1.0  1.0  1.0          9.90
10  1.0 -1.0  1.0 -1.0  1.0         18.34
11  1.0 -1.0  1.0  1.0 -1.0         15.57
12  1.0  1.0 -1.0 -1.0  1.0         25.71
13  1.0  1.0 -1.0  1.0 -1.0         28.24
14  1.0  1.0  1.0 -1.0 -1.0         39.42
15  1.0  1.0  1.0  1.0  1.0         27.72


In [3]:
candidate_formula = 'strength_MPa ~ A + B + C + D + E + A:B + C:D + B:C + A:E'
model_candidate = smf.ols(candidate_formula, data=df_adh).fit()
print(model_candidate.summary().tables[1])
print(f'\nResidual df: {int(model_candidate.df_resid)}  (16 runs - 9 candidate terms - 1 intercept)')

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     17.8194      0.249     71.518      0.000      17.210      18.429
A              4.1906      0.249     16.819      0.000       3.581       4.800
B              6.1769      0.249     24.791      0.000       5.567       6.787
C              2.9756      0.249     11.943      0.000       2.366       3.585
D             -1.5906      0.249     -6.384      0.001      -2.200      -0.981
E             -1.1056      0.249     -4.437      0.004      -1.715      -0.496
A:B            2.0856      0.249      8.371      0.000       1.476       2.695
C:D           -1.8244      0.249     -7.322      0.000      -2.434      -1.215
B:C           -0.2019      0.249     -0.810      0.449      -0.812       0.408
A:E           -0.4869      0.249     -1.954      0.099      -1.097       0.123

Residual df: 6  (16 runs - 9 candidate terms - 1 in

In [4]:
# ── Reuse Notebook 12 (Live Tutorial 2, Statistics)'s backward_elimination ─────────
def backward_elimination(data, response, candidate_terms, alpha=0.05, verbose=True):
    terms = list(candidate_terms)
    while True:
        formula = f'{response} ~ ' + ' + '.join(terms)
        fit = smf.ols(formula, data=data).fit()
        pvalues = fit.pvalues.drop('Intercept')
        worst_term, worst_p = pvalues.idxmax(), pvalues.max()
        if worst_p <= alpha or len(terms) == 1:
            return fit, terms
        if verbose:
            print(f'Dropping {worst_term!r} (p={worst_p:.3f})')
        terms.remove(worst_term)


candidates = ['A', 'B', 'C', 'D', 'E', 'A:B', 'C:D', 'B:C', 'A:E']
model_reduced, kept_terms = backward_elimination(df_adh, 'strength_MPa', candidates)
print('\nFinal terms kept:', kept_terms)
print(f'\nCandidate model: R2_adj={model_candidate.rsquared_adj:.3f}, AIC={model_candidate.aic:.1f}')
print(f'Reduced model:   R2_adj={model_reduced.rsquared_adj:.3f}, AIC={model_reduced.aic:.1f}')

Dropping 'B:C' (p=0.449)
Dropping 'A:E' (p=0.085)

Final terms kept: ['A', 'B', 'C', 'D', 'E', 'A:B', 'C:D']

Candidate model: R2_adj=0.988, AIC=49.6
Reduced model:   R2_adj=0.984, AIC=54.5


**Checkpoint**: backward elimination should have removed `B:C` and `A:E`
— exactly the two terms built with a true coefficient of zero in Section
24.2 — while keeping every genuinely non-zero term. This is model reduction
doing its job: a screening design's whole purpose is separating real
effects from a longer list of *plausible* ones, and this workflow makes
that separation defensible (p-values and AIC) rather than a judgement
call.

## 24.3 Fractional Factorial, Revisited Through a Reduction Lens

Notebook 18 covered aliasing in the abstract (defining relations, Resolution
III/IV/V). Model reduction adds a practical safety concern: **a term
backward elimination drops might not be truly zero — it might be aliased
with something that cancels it out**, or a term it *keeps* might actually
be estimating the sum of two confounded effects. Resolution V (used above)
keeps all main effects and 2-factor interactions alias-free of each other,
which is exactly why Section 24.2's reduction could trust each p-value at
face value. Drop to a coarser Resolution III design and that safety net
disappears — worth confirming directly.

In [5]:
# A Resolution III design for the same 5 factors: only 8 runs, but every
# 2-factor interaction is now aliased with some main effect
design_res3 = pyDOE3.fracfact('a b c ab ac')    # 2^(5-2), Resolution III: D=AB, E=AC
df_res3 = pd.DataFrame(design_res3, columns=['A', 'B', 'C', 'D', 'E'])
print(f'{len(df_res3)}-run Resolution III design for the same 5 factors (vs. {len(df_adh)} for Resolution V).')
print('\nAt Resolution III, main effects are aliased with 2-factor interactions --')
print('e.g. D and E here are literally defined as D=AB, E=AC, so a "D main effect"')
print('estimate is inseparably D + AB, not D alone. Always check a design\'s resolution')
print('before trusting that a "significant effect" is the effect its name suggests.')

8-run Resolution III design for the same 5 factors (vs. 16 for Resolution V).

At Resolution III, main effects are aliased with 2-factor interactions --
e.g. D and E here are literally defined as D=AB, E=AC, so a "D main effect"
estimate is inseparably D + AB, not D alone. Always check a design's resolution
before trusting that a "significant effect" is the effect its name suggests.


**Checkpoint / Exercise**: Fit `strength_MPa ~ A + B + C + D + E` on a
freshly simulated response for `df_res3` (reuse Section 24.2's true model,
substituting `df_res3`'s columns). Compare the fitted coefficient for `D`
to its true value (`-2`) — given `D` is aliased with `A:B` here, and `A:B`
has a real, nonzero true effect, in which direction would you expect the
estimate to be biased? Run it and check.

In [6]:
# ── Exercise: simulate a response on the Resolution III design and check the bias ──
true_strength_res3 = (18
                       + 4*df_res3.A + 6*df_res3.B + 3*df_res3.C - 2*df_res3.D - 1*df_res3.E
                       + 2*df_res3.A*df_res3.B - 1.5*df_res3.C*df_res3.D)
df_res3['strength_MPa'] = (true_strength_res3 + rng.normal(0, 1.0, len(df_res3))).round(2)

model_res3 = smf.ols('strength_MPa ~ A + B + C + D + E', data=df_res3).fit()
print(model_res3.params.round(3))
print(f"\nFitted D coefficient: {model_res3.params['D']:.3f}  (true D effect: -2.0)")
print(f"D is aliased with A:B (true A:B effect: +2.0) -- the fitted D coefficient should be")
print(f"pulled toward -2.0 + 2.0 = 0.0, i.e. biased upward (less negative) from the true -2.0.")

Intercept    17.704
A             4.176
B             6.024
C             3.729
D            -0.739
E            -1.034
dtype: float64

Fitted D coefficient: -0.739  (true D effect: -2.0)
D is aliased with A:B (true A:B effect: +2.0) -- the fitted D coefficient should be
pulled toward -2.0 + 2.0 = 0.0, i.e. biased upward (less negative) from the true -2.0.


:::{admonition} Take-home message
:class: tip

On the Resolution V design, backward elimination dropped exactly the two
null terms it should have — `B:C` (p=0.449) and `A:E` (p=0.085) — while
every genuinely non-zero candidate survived with its fitted coefficient
close to true (e.g. `A:B` 2.09 vs. true 2.0, `C:D` -1.82 vs. true -1.5).
On the Resolution III design, the fitted `D` coefficient came out at
**-0.739** — nowhere near its true value of **-2.0**, but also not simply
noisy: it landed almost exactly where the aliasing predicts. `D` is
literally `D=AB` on this design, so the fit can only ever report `D`'s
effect *plus* `A:B`'s effect (true value +2.0) as one inseparable number;
$-2.0 + 2.0 = 0.0$, and the observed $-0.739$ sits between that
alias-contaminated prediction and the clean $-2.0$ (with sampling noise
on top). **Resolution is not a detail you check after the fact — it
determines whether "the model said X" is even a meaningful sentence.**
:::

---
## Exercises

1. **Design efficiency comparison**: For the 5-factor adhesive study,
   compare the $2^{5-1}$ Resolution V design (16 runs, used in Section 24.2)
   to a $2^{5-2}$ Resolution III design (8 runs, Section 24.3) fit on the
   *same simulated system*. Does the Resolution III design still correctly
   identify `A`, `B`, `C`, `D`, `E` as significant main effects, and what
   happens to the `A:B` interaction — where does its effect "hide" once you
   can no longer estimate it separately?

2. **A different aliasing pattern**: Generate a second Resolution III
   design for the same 5 factors using a different set of generators (hint:
   `pyDOE3.fracfact('a b c bc ac')` or similar — check `pyDOE3`'s
   documentation for valid generator strings). Confirm it aliases `D` and
   `E` with *different* interactions than Section 24.3's design did, and
   repeat the bias check from the Checkpoint exercise for the new aliasing
   structure.

3. **Fold-over**: A folded-over Resolution III design (running a second
   fraction with all factor signs reversed) can separate main effects from
   their aliased 2-factor interactions. Generate the fold-over of Section
   24.3's design (`-1 * design_res3` in coded units), combine both halves
   into one 16-run dataset, and confirm the combined design now estimates
   `A:B` and `D` as fully separable effects again.